In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

C:\Users\Priyanshu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
seq_len = 8 
n_vocab = tokenizer.vocab_size
embed_dim = 128
nTransformerBlocks = 12
n_heads = 4
batch_size = 5

In [4]:
class MultiHeadAttention(nn.Module):
  def __init__(self, embed_dim, n_heads):
    super().__init__()
    assert embed_dim % n_heads == 0
    self.n_heads = n_heads
    self.head_dim = embed_dim // n_heads
    self.key   = nn.Linear(embed_dim, embed_dim, bias=False)
    self.query = nn.Linear(embed_dim, embed_dim, bias=False)
    self.value = nn.Linear(embed_dim, embed_dim, bias=False)
    self.W0    = nn.Linear(embed_dim, embed_dim, bias=False)

  def forward(self, x):
    B, T, C = x.size()
    k = self.key(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, T, head_dim)
    q = self.query(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    v = self.value(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    y = y.transpose(1, 2).contiguous().view(B, T, C)
    y = self.W0(y)
    return y

In [5]:
class TransformerBlock(nn.Module):
  def __init__(self, embed_dim, n_heads):
    super().__init__()
    self.layerNormAttn = nn.LayerNorm(embed_dim)
    self.attn = MultiHeadAttention(embed_dim, n_heads)
    self.layerNormMLP  = nn.LayerNorm(embed_dim)
    self.W1   = nn.Linear(embed_dim, 4*embed_dim) 
    self.gelu = nn.GELU()                        
    self.W2   = nn.Linear(4*embed_dim, embed_dim) 

  def forward(self, x):
    x = x + self.attn(self.layerNormAttn(x)) 
    y = x + self.W2(self.gelu(self.W1(self.layerNormMLP(x))))  
    return y

In [6]:
class LanguageModel(nn.Module):
  def __init__(self, nTransformerBlocks, embed_dim, n_heads):
    super().__init__()
    self.embedding = nn.Embedding(n_vocab, embed_dim)
    self.positions = nn.Embedding(seq_len, embed_dim)
    self.transformerBlocks = nn.Sequential(*[TransformerBlock(embed_dim, n_heads) for _ in range(nTransformerBlocks)])
    self.finalLayerNorm = nn.LayerNorm(embed_dim) 
    self.finalLinear = nn.Linear(embed_dim, n_vocab, bias=False)
    self.finalLinear.weight = nn.Parameter(self.embedding.weight)

  def forward(self, tokx):
    token_embed = self.embedding(tokx)
    posit_embed = self.positions(torch.arange(tokx.shape[-1]))
    x = token_embed + posit_embed
    x = self.transformerBlocks(x)
    x = self.finalLayerNorm(x)
    x = self.finalLinear(x)
    return x

  def generate(self, tokx, temperature=1., n_new_tokens=50):
    for _ in range(n_new_tokens):
      x = self(tokx[:, -seq_len:]) 
      x = x[:, -1, :]
      probs = F.softmax(x/temperature, dim=-1) 
      tokx_next = torch.multinomial(probs, num_samples=1)
      tokx = torch.cat( (tokx, tokx_next), dim=1)
    return tokx

In [7]:
llm = LanguageModel(nTransformerBlocks, embed_dim, n_heads)
llm

LanguageModel(
  (embedding): Embedding(50257, 128)
  (positions): Embedding(8, 128)
  (transformerBlocks): Sequential(
    (0): TransformerBlock(
      (layerNormAttn): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (key): Linear(in_features=128, out_features=128, bias=False)
        (query): Linear(in_features=128, out_features=128, bias=False)
        (value): Linear(in_features=128, out_features=128, bias=False)
        (W0): Linear(in_features=128, out_features=128, bias=False)
      )
      (layerNormMLP): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (W1): Linear(in_features=128, out_features=512, bias=True)
      (gelu): GELU(approximate='none')
      (W2): Linear(in_features=512, out_features=128, bias=True)
    )
    (1): TransformerBlock(
      (layerNormAttn): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (key): Linear(in_features=128, out_features=128, bias=Fals

In [8]:
llm.transformerBlocks[4]

TransformerBlock(
  (layerNormAttn): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (attn): MultiHeadAttention(
    (key): Linear(in_features=128, out_features=128, bias=False)
    (query): Linear(in_features=128, out_features=128, bias=False)
    (value): Linear(in_features=128, out_features=128, bias=False)
    (W0): Linear(in_features=128, out_features=128, bias=False)
  )
  (layerNormMLP): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (W1): Linear(in_features=128, out_features=512, bias=True)
  (gelu): GELU(approximate='none')
  (W2): Linear(in_features=512, out_features=128, bias=True)
)

In [9]:
tokens = tokenizer.encode('I prefer oat milk in my coffee.')
X = torch.tensor(tokens[:-1]).unsqueeze(0)
y = torch.tensor(tokens[1:]).unsqueeze(0)

print(X.shape)
print(y.shape)

torch.Size([1, 8])
torch.Size([1, 8])


In [10]:
out = llm(X)
print(out.shape)

torch.Size([1, 8, 50257])
